# Notebook 02 — Fine-tune Student Model

**Member responsible: B**

LoRA fine-tuning for Exp 2, 3, and 4. Switch experiments by editing `configs/finetune.yaml`.

**Before running:** Runtime > Change runtime type > **T4 GPU**

| Experiment | `data.mode` | `training.output_dir` |
|---|---|---|
| Exp 2 (A1 adaptation)   | `adaptation`   | `results/checkpoints/adaptation` |
| Exp 3 (D1 distillation) | `distillation` | `results/checkpoints/distillation` |
| Exp 4 (D2 filtered)     | `filtered`     | `results/checkpoints/filtered` |

**Ablations (change one field per run):**
- A2 rank sweep: `lora.rank` ∈ {4, 8, 16, 32} — update `output_dir` each time
- A3 data size:  `data.train_fraction` ∈ {0.1, 0.25, 0.5, 1.0}

In [ ]:
# Step 1a: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 1b: Clone repo and install dependencies
# Paste your GitHub Personal Access Token below (never commit this notebook with the token filled in)
# Generate one at: GitHub > Settings > Developer settings > Personal access tokens > Tokens (classic) > repo scope
import os
token = 'ghp_xxxxxxxxxxxx'  # TODO: paste your token here each session
if not os.path.exists('/content/SC4001-Group133'):
    os.system(f'git clone https://{token}@github.com/TIFFMUN/SC4001-Group133.git /content/SC4001-Group133')
%cd /content/SC4001-Group133
!pip install -r requirements.txt -q

In [ ]:
# Step 1c: Redirect HF cache + data/ + results/ to Drive
# MUST run before any HuggingFace imports
# This also makes distillation datasets (built in notebook 01) immediately available
# because data/ is symlinked to Drive — no manual copying needed
import sys
sys.path.insert(0, '/content/SC4001-Group133')
from src.setup_cache import setup
setup()

In [ ]:
# Step 2: Check GPU
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('No GPU detected. Enable GPU: Runtime > Change runtime type > T4 GPU')

In [ ]:
# Step 3: Check current config before training
import yaml
with open('configs/finetune.yaml') as f:
    cfg = yaml.safe_load(f)

print(f"Mode:           {cfg['data']['mode']}")
print(f"LoRA rank:      {cfg['lora']['rank']}")
print(f"Train fraction: {cfg['data']['train_fraction']}")
print(f"Output dir:     {cfg['training']['output_dir']}")
print(f"Epochs:         {cfg['training']['num_epochs']}")
# TODO: edit configs/finetune.yaml directly to switch experiments before running next cell

In [ ]:
# Step 4: Run training
# Checkpoints save every 200 steps to results/checkpoints/ which is symlinked to Drive
# If session dies: re-run setup cells (1a-1c), then re-run this cell — resumes from last checkpoint
# Expected time on T4: ~2-4 hours for full dataset, 3 epochs
from src.training.finetune import train
train()

In [ ]:
# Step 5: Confirm checkpoint saved to Drive
import os
output_dir = cfg['training']['output_dir']
drive_path = f"/content/drive/MyDrive/SC4001/{output_dir}"

# Because results/ is symlinked to Drive, the checkpoint is already there
if os.path.exists(drive_path):
    files = os.listdir(drive_path)
    print(f'Checkpoint saved at {drive_path}')
    print(f'Files: {files}')
else:
    print(f'WARNING: checkpoint not found at {drive_path} — check output_dir in finetune.yaml')